# Lecture 5: From ResNet to ConvNeXt

- INFO8010 Deep Learning, Gilles Louppe**
- Lecture 5: [Convolutional neural networks](https://glouppe.github.io/info8010-deep-learning/?p=lecture5.md)

This notebook starts from a basic ResNet and progressively modernizes it toward ConvNeXt, following the ablation in [Liu et al. (2022)](https://arxiv.org/abs/2201.03545). Each step introduces one architectural change, showing how decades of CNN design can be distilled into a few key ideas borrowed from transformers.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import time

plt.rcParams['figure.dpi'] = 100
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

## CIFAR-10

In [ ]:
train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
])

train_data = datasets.CIFAR10('./data', train=True, download=True, transform=train_transform)
test_data = datasets.CIFAR10('./data', train=False, transform=test_transform)
train_loader = DataLoader(train_data, batch_size=128, shuffle=True, num_workers=0)
test_loader = DataLoader(test_data, batch_size=256)

## Training infrastructure

We keep the training loop fixed across all experiments so differences come purely from the architecture.

In [ ]:
def count_params(model):
    return sum(p.numel() for p in model.parameters()) / 1e3

def train_and_eval(model, n_epochs=10, lr=1e-3):
    model = model.to(device)
    print(f"Parameters: {count_params(model):.1f}K")

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-2)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)

    train_losses, test_accs = [], []

    for epoch in range(n_epochs):
        model.train()
        total_loss = 0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            loss = F.cross_entropy(model(x), y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        train_losses.append(total_loss / len(train_loader))
        scheduler.step()

        model.eval()
        correct = 0
        with torch.no_grad():
            for x, y in test_loader:
                x, y = x.to(device), y.to(device)
                correct += (model(x).argmax(1) == y).sum().item()
        acc = correct / len(test_data)
        test_accs.append(acc)

        if (epoch + 1) % 5 == 0:
            print(f"  epoch {epoch+1:2d}: loss={train_losses[-1]:.3f}, acc={acc:.4f}")

    return train_losses, test_accs

## Step 0: ResNet baseline

A standard ResNet with two residual blocks per stage, batch normalization, and ReLU. This is our starting point.

In [ ]:
class ResBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(channels),
            nn.ReLU(),
            nn.Conv2d(channels, channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(channels),
        )

    def forward(self, x):
        x = x + self.net(x)
        x = F.relu(x)
        return x

class ResNet(nn.Module):
    def __init__(self, channels=32):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(channels),
            nn.ReLU())

        self.stage1 = nn.Sequential(
            ResBlock(channels),
            ResBlock(channels))

        self.stage2 = nn.Sequential(
            nn.Conv2d(channels, channels * 2, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(channels * 2),
            nn.ReLU(),
            ResBlock(channels * 2),
            ResBlock(channels * 2))

        self.stage3 = nn.Sequential(
            nn.Conv2d(channels * 2, channels * 4, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(channels * 4),
            nn.ReLU(),
            ResBlock(channels * 4),
            ResBlock(channels * 4))

        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(channels * 4, 10))

    def forward(self, x):
        x = self.stem(x)
        x = self.stage1(x)
        x = self.stage2(x)
        x = self.stage3(x)
        return self.head(x)

results = {}
results['ResNet'] = train_and_eval(ResNet())

## Step 1: Inverted bottleneck

ResNet blocks use equal-width layers. Transformers use a 4x expansion in the FFN. We adopt the same idea: narrow → wide → narrow, using a 1x1 → depthwise 3x3 → 1x1 pattern.

In [ ]:
class InvertedBlock(nn.Module):
    def __init__(self, channels, expansion=4):
        super().__init__()
        mid = channels * expansion
        self.net = nn.Sequential(
            nn.Conv2d(channels, mid, 1, bias=False),
            nn.BatchNorm2d(mid),
            nn.ReLU(),
            nn.Conv2d(mid, mid, 3, padding=1, groups=mid, bias=False),  # depthwise
            nn.BatchNorm2d(mid),
            nn.ReLU(),
            nn.Conv2d(mid, channels, 1, bias=False),
            nn.BatchNorm2d(channels),
        )

    def forward(self, x):
        x = x + self.net(x)
        x = F.relu(x)
        return x

class ResNetV2(nn.Module):
    def __init__(self, channels=32):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(channels),
            nn.ReLU())

        self.stage1 = nn.Sequential(
            InvertedBlock(channels),
            InvertedBlock(channels))

        self.stage2 = nn.Sequential(
            nn.Conv2d(channels, channels * 2, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(channels * 2),
            nn.ReLU(),
            InvertedBlock(channels * 2),
            InvertedBlock(channels * 2))

        self.stage3 = nn.Sequential(
            nn.Conv2d(channels * 2, channels * 4, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(channels * 4),
            nn.ReLU(),
            InvertedBlock(channels * 4),
            InvertedBlock(channels * 4))

        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(channels * 4, 10))

    def forward(self, x):
        x = self.stem(x)
        x = self.stage1(x)
        x = self.stage2(x)
        x = self.stage3(x)
        return self.head(x)

results['+ inverted bottleneck'] = train_and_eval(ResNetV2())

## Step 2: Larger kernel

Standard CNNs use 3x3 kernels. Transformers have global receptive fields via attention. A compromise: use 7x7 depthwise convolutions. We also move the depthwise conv to the beginning of the block (as in Swin Transformer's spatial mixing before channel mixing).

In [ ]:
class LargeKernelBlock(nn.Module):
    def __init__(self, channels, expansion=4):
        super().__init__()
        mid = channels * expansion
        self.net = nn.Sequential(
            nn.Conv2d(channels, channels, 7, padding=3, groups=channels, bias=False),  # depthwise 7x7
            nn.BatchNorm2d(channels),
            nn.ReLU(),
            nn.Conv2d(channels, mid, 1, bias=False),
            nn.BatchNorm2d(mid),
            nn.ReLU(),
            nn.Conv2d(mid, channels, 1, bias=False),
            nn.BatchNorm2d(channels),
        )

    def forward(self, x):
        x = x + self.net(x)
        x = F.relu(x)
        return x

class ResNetV3(nn.Module):
    def __init__(self, channels=32):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(channels),
            nn.ReLU())

        self.stage1 = nn.Sequential(
            LargeKernelBlock(channels),
            LargeKernelBlock(channels))

        self.stage2 = nn.Sequential(
            nn.Conv2d(channels, channels * 2, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(channels * 2),
            nn.ReLU(),
            LargeKernelBlock(channels * 2),
            LargeKernelBlock(channels * 2))

        self.stage3 = nn.Sequential(
            nn.Conv2d(channels * 2, channels * 4, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(channels * 4),
            nn.ReLU(),
            LargeKernelBlock(channels * 4),
            LargeKernelBlock(channels * 4))

        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(channels * 4, 10))

    def forward(self, x):
        x = self.stem(x)
        x = self.stage1(x)
        x = self.stage2(x)
        x = self.stage3(x)
        return self.head(x)

results['+ large kernel (7x7)'] = train_and_eval(ResNetV3())

## Step 3: LayerNorm + GELU + fewer activations

Replace BatchNorm with LayerNorm and ReLU with GELU, matching transformer conventions. LayerNorm normalizes per-sample (no batch dependency), and GELU provides a smoother activation.

We also reduce the number of activations to just one (between the two 1x1 layers), and remove the post-addition activation on the residual connection. This mirrors the transformer MLP block, which uses a single activation.

In [ ]:
class ConvNeXtBlock(nn.Module):
    def __init__(self, channels, expansion=4):
        super().__init__()
        mid = channels * expansion
        self.net = nn.Sequential(
            nn.Conv2d(channels, channels, 7, padding=3, groups=channels),  # depthwise 7x7
            nn.GroupNorm(1, channels),  # LayerNorm for conv (groups=1)
            nn.Conv2d(channels, mid, 1),
            nn.GELU(),
            nn.Conv2d(mid, channels, 1),
        )

    def forward(self, x):
        x = x + self.net(x)
        return x 

class ConvNeXt(nn.Module):
    def __init__(self, channels=32):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, channels, 4, stride=1, padding=1),  # patchify stem (simplified for 32x32)
            nn.GroupNorm(1, channels))

        self.stage1 = nn.Sequential(
            ConvNeXtBlock(channels),
            ConvNeXtBlock(channels))

        self.down1 = nn.Sequential(
            nn.GroupNorm(1, channels),
            nn.Conv2d(channels, channels * 2, 2, stride=2))

        self.stage2 = nn.Sequential(
            ConvNeXtBlock(channels * 2),
            ConvNeXtBlock(channels * 2))

        self.down2 = nn.Sequential(
            nn.GroupNorm(1, channels * 2),
            nn.Conv2d(channels * 2, channels * 4, 2, stride=2))

        self.stage3 = nn.Sequential(
            ConvNeXtBlock(channels * 4),
            ConvNeXtBlock(channels * 4))

        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.GroupNorm(1, channels * 4),  # final norm before classifier
            nn.Linear(channels * 4, 10))

    def forward(self, x):
        x = self.stem(x)
        x = self.down1(self.stage1(x))
        x = self.down2(self.stage2(x))
        x = self.stage3(x)
        return self.head(x)

results['ConvNeXt'] = train_and_eval(ConvNeXt())

## Comparison

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for name, (losses, accs) in results.items():
    ax1.plot(losses, label=name)
    ax2.plot([a * 100 for a in accs], label=name)

ax1.set_xlabel("Epoch"); ax1.set_ylabel("Training loss"); ax1.legend(); ax1.set_title("Training loss")
ax2.set_xlabel("Epoch"); ax2.set_ylabel("Test accuracy (%)"); ax2.legend(); ax2.set_title("Test accuracy")
plt.show()

In [ ]:
for name, (losses, accs) in results.items():
    print(f"{name:30s}  best acc: {max(accs)*100:.1f}%")

## Takeaway

Starting from a standard ResNet, we applied four modernizations inspired by transformers:

1. Inverted bottleneck (narrow → wide → narrow, with depthwise separable convolutions)
2. Larger kernels (7x7 depthwise convolutions for wider receptive fields)
3. LayerNorm instead of BatchNorm (per-sample normalization)
4. GELU instead of ReLU (smoother activation)

Each change is simple, but together they close the gap between CNNs and vision transformers. The key insight from [ConvNeXt (Liu et al., 2022)](https://arxiv.org/abs/2201.03545): much of the transformer advantage comes from training recipes and micro-design choices, not from the attention mechanism itself.